# Bullish and Bearish Credit Assessment

Run these cells in order with the project's Python 3.14 kernel. This notebook is
standalone; no variables or execution from notebooks 01–03 are required.

We calculate one synthetic borrower's covenant assessment, compare independent
**bullish** and **bearish** interpretations, then inspect the **recommendation**
agent's judgment of both cases. The application runs the two perspectives concurrently
against the same frozen evidence, then gives their full analyses and the original
evidence to the recommendation agent. Review agents have no calculation tools;
they interpret the service's results and cannot change compliance.

Before starting, run `uv sync --group notebooks --group dev --locked` from the repository
root and select the project's `.venv` kernel. Sections 1–4 run without model credentials.
The review call in section 5 uses the OpenAI API and requires `OPENAI_API_KEY` in the
environment or root `.env` file. Model calls incur API usage.

This walkthrough uses local synthetic inputs, with **ML disabled** and no public news
retrieval. It needs neither an ML artifact nor SEC downloads. Recommendations are
advisory; an analyst makes the exposure decision.

## 1. Set up paths and configuration

Start Jupyter from the repository root or `homework/notebooks/`. Imported project
paths work in either location. Edit the borrower and dates here before calculating.
The cutoff must be on or after the reporting date; source availability warnings remain
visible even when calculation verification passes.

In [ ]:
from datetime import date

import pandas as pd
from dotenv import load_dotenv
from IPython.display import JSON, Markdown, display

from credit_monitoring.agents.models import DEFAULT_MODEL
from credit_monitoring.observability import configure_observability
from credit_monitoring.observability.logging import configure_logging
from credit_monitoring.paths import APP_SYNTHETIC_DIR, ROOT, RUNTIME_DIR

load_dotenv(ROOT / ".env")
configure_observability()
configure_logging()

MODEL = DEFAULT_MODEL
BORROWER_ID = "SYN002"
PERIOD_END = date(2025, 9, 30)
INFORMATION_CUTOFF = date(2025, 9, 30)

# Dedicated local state, separate from notebook 03 and the app workspace.
review_dir = RUNTIME_DIR / "notebooks" / "credit_review"
review_dir.mkdir(parents=True, exist_ok=True)

print("Repository:", ROOT)
print("Review model:", MODEL)
print("Borrower:", BORROWER_ID, "| Period:", PERIOD_END, "| Cutoff:", INFORMATION_CUTOFF)
print("Runtime storage:", review_dir)

## 2. Load synthetic financials and create the assessment service

`load_financials` upserts the demo snapshots into a dedicated financial database.
Calculated results use a separate analytics database. These cells make no model calls.
Synthetic borrower IDs have no inferred relationship to real SEC issuers.

In [ ]:
from credit_monitoring.application.bootstrap import (
    create_assessment_service,
    create_credit_review_service,
)
from credit_monitoring.config.settings import AssessmentSettings
from credit_monitoring.covenants.structured import SyntheticCovenantReader
from credit_monitoring.financials.repositories import FinancialRepository
from credit_monitoring.ingestion.loaders.financials import load_financials

financial_db = review_dir / "financials.sqlite3"
load_result = load_financials(
    APP_SYNTHETIC_DIR / "synthetic_quarterly_financials.csv", financial_db
)
print(load_result)

assessment_service = create_assessment_service(
    financial_repository=FinancialRepository(financial_db),
    covenant_reader=SyntheticCovenantReader(APP_SYNTHETIC_DIR / "synthetic_covenant_terms.csv"),
    settings=AssessmentSettings(
        analytics_database=review_dir / "analytics.sqlite3",
        ml_enabled=False,
    ),
)

## 3. Calculate and inspect one assessment

This call creates a new assessment run, including available history and comparable
trends. ML remains disabled: no probability or forecast is requested. The review will
reuse this exact result rather than calculate another assessment.

Headroom is direction-aware and uses the covenant's unit. A blank value is unavailable,
not zero. Quarter-on-quarter changes below come from the application; unavailable or
incompatible comparisons remain missing.

In [ ]:
assessment = assessment_service.assess(
    BORROWER_ID,
    period_end=PERIOD_END,
    information_cutoff=INFORMATION_CUTOFF,
    ml_enabled=False,
)

print("Assessment run:", assessment.assessment_run_id)
print("ML status:", assessment.ml_status)
print("Verification valid:", assessment.verification.is_valid)
display(pd.DataFrame([
    {
        "result_id": result.result_id,
        "covenant": result.covenant_id,
        "status": result.compliance_status,
        "actual": result.actual_value,
        "operator": result.operator,
        "threshold": result.threshold,
        "unit": result.unit,
        "headroom": result.headroom,
        "issues": "; ".join(result.issues),
    }
    for result in assessment.results
]))

if assessment.trends is not None:
    display(Markdown("### Covenant trends"))
    display(pd.DataFrame([
        trend.model_dump() for trend in assessment.trends.covenants
    ]))
    display(Markdown("### Borrower trends and information availability"))
    display(JSON(assessment.trends.model_dump(mode="json", exclude={"covenants"})))
else:
    print("Comparable trends are unavailable.")

if assessment.verification.issues:
    display(pd.DataFrame([
        issue.model_dump() for issue in assessment.verification.issues
    ]))
for issue in assessment.issues:
    print("Assessment issue:", issue)

## 4. Supply the decision question and analyst criteria

These are **illustrative analyst criteria**, not contractual rules or a production
credit policy. Edit them to ask a different exposure question. They guide interpretation;
they do not override the calculated results. The service permits `defer` when support
is insufficient, even if the question requests a direction.

In [ ]:
from credit_monitoring.domain.credit_review import ExposureDecisionContext

decision_context = ExposureDecisionContext(
    question=(
        f"For {BORROWER_ID}, should we maintain, increase, reduce, or review the existing "
        f"exposure for {PERIOD_END}, using information cutoff {INFORMATION_CUTOFF}?"
    ),
    credit_criteria=(
        "Evaluate current covenant compliance and remaining headroom.",
        "Weigh comparable deterioration or improvement in headroom and financial trends.",
        "Account for missing inputs, verification warnings, and uncertain source availability.",
        "Identify the strongest supported case on each side and what would change the view.",
    ),
    constraints=(
        "Give an advisory direction without numerical exposure sizing.",
        "Preserve service-calculated compliance; do not invent a forecast or missing evidence.",
    ),
)
display(JSON(decision_context.model_dump(mode="json")))

## 5. Run the bullish, bearish, and recommendation agents

Construction makes no model requests. The next call requires `OPENAI_API_KEY` and
makes model requests: bullish and bearish reviews run concurrently, followed by the
recommendation. We use the factory's default timeout (60 seconds per role) and one
output-validation retry. Progress reports stage events, not private model reasoning;
the order of bullish and bearish completion can vary.

The default news provider performs no search and reports `unavailable`. That status
does not establish an absence of relevant news.

In [ ]:
async def show_review_progress(event):
    detail = f" | error={event.error_code}" if event.error_code else ""
    print(f"{event.stage}: {event.kind}{detail}", flush=True)


review_service = create_credit_review_service(
    model=MODEL,
    event_handler=show_review_progress,
)

In [ ]:
# Live API calls: rerun this cell to review the same assessment again.
review = await review_service.review_assessment_async(
    assessment,
    decision_context=decision_context,
)

assert review.snapshot.assessment_run_id == assessment.assessment_run_id
print("Review:", review.review_id)
print("Assessment run:", review.snapshot.assessment_run_id)
print("Review status:", review.status)
print("Public news:", review.snapshot.news.status)

## 6. Compare the bullish and bearish cases

Each case includes a thesis, cited arguments and rationales, counterevidence,
assumptions, gaps, and conditions that would change the view. A bullish perspective
can acknowledge weaknesses; a bearish perspective can acknowledge strengths.
Evidence confidence describes support for the analysis, not breach probability.

If a reviewer fails, its analysis is unavailable. Preserve the other review and inspect
the reported error rather than treating missing analysis as agreement.

In [ ]:
def show_items(label, items):
    display(Markdown(f"**{label}**"))
    if items:
        display(Markdown("\n".join(f"- {item}" for item in items)))
    else:
        print("None reported.")


def show_perspective(perspective):
    display(Markdown(f"### {perspective.perspective.title()} case"))
    print("Execution:", perspective.execution.status)
    analysis = perspective.analysis
    if analysis is None:
        print("Analysis unavailable:", perspective.execution.error_code or "See review issues.")
        return
    display(Markdown(analysis.thesis))
    print("Evidence confidence:", analysis.confidence)
    display(Markdown(analysis.confidence_explanation))
    if not analysis.arguments:
        print("No supported arguments reported.")
    for argument in analysis.arguments:
        display(Markdown(f"**{argument.argument_id}: {argument.statement}**"))
        display(Markdown(argument.rationale))
        display(Markdown(f"Credit implication: {argument.credit_implication}"))
        print("Evidence IDs:", ", ".join(argument.evidence_ids))
        show_items("Argument assumptions", argument.assumptions)
    display(Markdown("**Counterevidence**"))
    if not analysis.counterevidence:
        print("None reported.")
    for claim in analysis.counterevidence:
        display(Markdown(claim.statement))
        display(Markdown(claim.rationale))
        print("Evidence IDs:", ", ".join(claim.evidence_ids))
    show_items("Perspective assumptions", analysis.assumptions)
    show_items("Evidence gaps", analysis.gaps)
    show_items("What would change this view", analysis.change_of_view_conditions)


show_perspective(review.bullish)
show_perspective(review.bearish)

## 7. Inspect the combined recommendation

The recommendation judges every supplied argument as `accept`, `qualify`, or `reject`
and explains its action. Actions are `maintain`, `increase`, `reduce`, `review`, or
`defer`; none executes a credit decision.

`complete` means the workflow finished and can still recommend `defer`. Verification
or identity errors produce a blocked review and skip the agents. A failed reviewer or
recommendation produces an incomplete review with deferral and retains available work.
Read the issues and evidence limitations together with the recommendation.

In [ ]:
recommendation = review.recommendation
print("Review status:", review.status)
print("Advisory action:", recommendation.action)
display(Markdown(recommendation.rationale))
print("Evidence confidence:", recommendation.confidence)
display(Markdown(recommendation.confidence_explanation))
print("Evidence IDs:", ", ".join(recommendation.evidence_ids) or "None reported.")

if recommendation.argument_judgments:
    display(pd.DataFrame([
        judgment.model_dump() for judgment in recommendation.argument_judgments
    ]))
else:
    print("No argument judgments available.")

show_items("Conditions", recommendation.conditions)
show_items("Diligence questions", recommendation.diligence_questions)
show_items("Unresolved uncertainties", recommendation.unresolved_uncertainties)
show_items("Review issues and evidence limitations", review.issues)

## 8. Trace claims to evidence and inspect execution

Argument and recommendation evidence IDs resolve through the shared registry below.
The JSON pointers refer to `review.snapshot.prompt_context()`: calculated results retain
financial inputs and source provenance. Generated narratives and source notes are
excluded from this snapshot. Reference validation checks IDs; it does not establish
that every generated statement is supported. Inspect the cited data yourself.

In [ ]:
display(pd.DataFrame([
    reference.model_dump() for reference in review.snapshot.evidence
]))


def inspect_evidence(evidence_id):
    reference = next(
        item for item in review.snapshot.evidence if item.evidence_id == evidence_id
    )
    value = review.snapshot.prompt_context()
    for part in reference.path.lstrip("/").split("/"):
        key = part.replace("~1", "/").replace("~0", "~")
        value = value[int(key)] if isinstance(value, list) else value[key]
    return value


# Replace this with any cited evidence ID from either case or the recommendation.
EVIDENCE_ID = next(
    (item.evidence_id for item in review.snapshot.evidence if item.kind == "calculation"),
    "assessment",
)
print("Selected evidence:", EVIDENCE_ID)
display(JSON(inspect_evidence(EVIDENCE_ID)))

In [ ]:
executions = (
    review.bullish.execution,
    review.bearish.execution,
    review.recommendation_execution,
)
display(pd.DataFrame([
    execution.model_dump() for execution in executions
]))

# Optional: inspect the full structured review, including all claims and provenance.
# display(JSON(review.model_dump(mode="json")))

## 9. Try another decision or borrower

- **Change the question or criteria:** edit section 4, rerun it, then rerun the review
  call in section 5 and displays in sections 6–8. This reuses the same assessment run.
- **Change the borrower or dates:** edit section 1, rerun sections 1–8. Section 3 creates
  a new assessment run; prior results remain in the analytics database.
- **Inspect a different claim:** copy its evidence ID into `EVIDENCE_ID` in section 8.
  Inspect its supporting inputs, provenance, and any availability warnings.
- **Explore evidence gaps:** try `SYN007` with the same dates to examine conflicting
  debt inputs. An incomplete covenant result need not block the review workflow;
  retain its unresolved status and inspect the agents' conclusions.

Interactive follow-up conversations are demonstrated in [notebook 03](03-agents.ipynb).
For the service interfaces and review boundaries, see the
[credit review guide](../../docs/guides/credit_review.md).